# Exploración y pruebas del RAG de normativa de tránsito

**Trabajo Integrador 2 · Laboratorio de PLN · IFTS N.º 24**

## Objetivo

Comprobar, con evidencia propia, qué extrajo el cargador de cada PDF, cómo cambia la recuperación según el corte y qué hace el sistema con preguntas respondibles y no respondibles. Este cuaderno hace `import app`: usa las mismas constantes y las mismas funciones que la aplicación, de modo que lo que se prueba es el sistema real y no una copia.

## Cómo usarlo

* Ejecutarlo desde la carpeta del proyecto (la misma de `app.py`), con el kernel del `.venv` y Ollama abierto.
* Ejecutar las celdas en orden.
* Si se modifica `app.py`, se debe reiniciar el kernel para que `import app` tome el cambio.

## Criterios de comprobación

* Señalar el archivo y la página original de cada fragmento que llega al modelo.
* Revisar el texto extraído de cada PDF contra el original.
* Comparar dos formas de fragmentar con la misma pregunta.
* Escribir los casos de prueba antes de ejecutarlos.
* Distinguir una falla de extracción, de corte, de recuperación o de generación.

## 1. Cargar antes de indexar


In [1]:
import statistics
from collections import Counter
from pathlib import Path

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

import app  # mismas constantes y mismo pipeline que la aplicación

print("Archivos del corpus:", app.ARCHIVOS)
print("Modelo de embeddings:", app.MODELO_EMBEDDINGS)
print("Modelo generador:", app.MODELO_GENERADOR)
print("Corte de app.py:", app.TAMANO_FRAGMENTO, "/", app.SOLAPAMIENTO, "| k =", app.K)

C:\Users\FAF\AppData\Local\Temp\ipykernel_14768\3068956863.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
e:\Tecnicatura Cs de Dts\Cursadas\2do año\4to Cuatrimestre\Tecnol de Procesamiento del Lenguaje Natural\TP FINAL PLN - RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Archivos del corpus: ['ley_24449_transito.pdf', 'decreto_779_1995_reglamentario.pdf', 'decreto_242_2022_scoring.pdf', 'anexo2_decreto_779_1995.pdf']
Modelo de embeddings: bge-m3
Modelo generador: granite4:latest
Corte de app.py: 1000 / 200 | k = 3


In [ ]:
# Cargar cada PDF por separado para ver cuánto texto extrajo de cada uno
paginas_por_archivo = {}

for nombre in app.ARCHIVOS:
    ruta = app.RAIZ / "datos" / nombre
    assert ruta.exists(), f"No encontré {ruta}. Revisá los nombres en datos/."
    paginas_por_archivo[nombre] = PyPDFLoader(str(ruta)).load()

for nombre, paginas in paginas_por_archivo.items():
    caracteres = []

    for pagina in paginas:
        caracteres.append(len(pagina.page_content.strip()))

    pocas = []

    for numero in range(len(paginas)):
        if caracteres[numero] < 200:
            pocas.append(paginas[numero].metadata["page"] + 1)

    print(f"\n{nombre}")
    print("  Páginas:", len(paginas))
    print("  Caracteres por página (mín / mediana / máx):",
          min(caracteres), "/", int(statistics.median(caracteres)), "/", max(caracteres))
    print("  Páginas con menos de 200 caracteres:", pocas)


ley_24449_transito.pdf
  Páginas: 69
  Caracteres por página (mín / mediana / máx): 904 / 2088 / 3163
  Páginas con menos de 200 caracteres: []

decreto_779_1995_reglamentario.pdf
  Páginas: 5
  Caracteres por página (mín / mediana / máx): 132 / 1584 / 2461
  Páginas con menos de 200 caracteres: [5]

decreto_242_2022_scoring.pdf
  Páginas: 4
  Caracteres por página (mín / mediana / máx): 1694 / 2304 / 2867
  Páginas con menos de 200 caracteres: []

anexo2_decreto_779_1995.pdf
  Páginas: 16
  Caracteres por página (mín / mediana / máx): 1678 / 3071 / 3355
  Páginas con menos de 200 caracteres: []


Una página con muy pocos caracteres puede ser una portada, una página casi en blanco o contenido que no es texto (imagen, escaneo). Una página con 0 caracteres no aporta nada al índice.

In [ ]:
# Revisión manual: elegir un archivo y una página (contada desde 1, como en el visor de PDF)
ARCHIVO_A_REVISAR = app.ARCHIVOS[3]
PAGINA_A_REVISAR = 1

pagina = paginas_por_archivo[ARCHIVO_A_REVISAR][PAGINA_A_REVISAR - 1]
assert pagina.metadata["page"] + 1 == PAGINA_A_REVISAR

print("Metadatos:", pagina.metadata)
print("Caracteres extraídos:", len(pagina.page_content))
print("\nTexto extraído:\n")
print(pagina.page_content)

Metadatos: {'producer': 'Skia/PDF m149', 'creator': 'Firefox', 'creationdate': '2026-11-13T03:32:17+00:00', 'title': 'Texto actualizado del ANEXO 2 - Decreto Reglamentario 779/1995 - TRANSITO Y SEGURIDAD VIAL | Argentina.gob.ar', 'moddate': '2026-11-13T03:32:17+00:00', 'source': 'E:\\Tecnicatura Cs de Dts\\Cursadas\\2do año\\4to Cuatrimestre\\Tecnol de Procesamiento del Lenguaje Natural\\TP FINAL PLN - RAG\\datos\\anexo2_decreto_779_1995.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}
Caracteres extraídos: 1678

Texto extraído:

Texto actualizado
 (
/normativa/nacional/norma-30389/actualizacion
)
/
Decreto Reglamentario 779/1995
PODER EJECUTIVO NACIONAL (P.E.N.)
TRANSITO Y SEGURIDAD VIAL
REGLAMENTACION
ANEXO 2
(Anexo sustituido por art. 1º del
Decreto Nº 242/2022
 (
/normativa/nacional/norma-364297
)
B.O. 6/5/2022. Vigencia: a partir de los SESENTA (60) días desde su publicación en el
BOLETÍN OFICIAL.)
RÉGIMEN DE CONTRAVENCIONES Y SANCIONES POR FALTAS COMETIDAS A LA LEY N°
24.44

Abrir el PDF original en esa misma página y comparar. Fijarse en: columnas o tablas que se hayan mezclado, artículos cortados, numeración, notas al pie, enlaces y texto que no pertenece a la norma. Se repite con una página de cada PDF (cambiando `ARCHIVO_A_REVISAR` y `PAGINA_A_REVISAR`).

In [10]:
# Líneas que se repiten en muchas páginas: suelen ser encabezados o pies del navegador
for nombre, paginas in paginas_por_archivo.items():
    conteo = Counter()

    for pagina in paginas:
        lineas = set()

        for linea in pagina.page_content.splitlines():
            linea = linea.strip()

            if linea:
                lineas.add(linea)

        conteo.update(lineas)

    minimo = max(2, len(paginas) // 2)
    print(f"\n{nombre} (líneas presentes en {minimo} páginas o más)")
    hubo = False

    for linea, veces in conteo.most_common(10):
        if veces >= minimo:
            print(f"  {veces:>3} páginas: {linea[:110]}")
            hubo = True

    if not hubo:
        print("  Ninguna.")


ley_24449_transito.pdf (líneas presentes en 34 páginas o más)
   39 páginas: (
   39 páginas: )

decreto_779_1995_reglamentario.pdf (líneas presentes en 2 páginas o más)
    3 páginas: (
    3 páginas: )
    2 páginas: Anexo C
    2 páginas: Decreto N° 32/2018
    2 páginas: B.O.  18/3/2025.
    2 páginas: Decreto N° 196/2025
    2 páginas: Anexo K
    2 páginas: Anexo G
    2 páginas: Anexo 1
    2 páginas: Anexo B

decreto_242_2022_scoring.pdf (líneas presentes en 2 páginas o más)
    2 páginas: (

anexo2_decreto_779_1995.pdf (líneas presentes en 8 páginas o más)
  Ninguna.


## 2. Comparar dos formas de fragmentar

Mismo corpus, misma pregunta, dos cortes. `chunk_size` y `chunk_overlap` se miden en caracteres. 

| Versión | `chunk_size` | `chunk_overlap` |
|---|---:|---:|
| Breve | 500 | 100 |
| Amplio | el de `app.py` | el de `app.py` |



In [ ]:
BREVE = (500, 100)
AMPLIO = (app.TAMANO_FRAGMENTO, app.SOLAPAMIENTO)  # el corte actual de app.py
ARCHIVOS_PARA_COMPARAR = app.ARCHIVOS  # limitar la lista si tarda demasiado

PREGUNTA_COMPARACION = "en una encrucijada de calles, qué vehículo tiene prioridad de paso? "
assert "COMPLETAR" not in PREGUNTA_COMPARACION, "Escribí tu pregunta antes de ejecutar."

documentos = []

for nombre in ARCHIVOS_PARA_COMPARAR:
    documentos.extend(paginas_por_archivo[nombre])

embeddings = OllamaEmbeddings(model=app.MODELO_EMBEDDINGS)
indices_temporales = {}

for etiqueta, (tamano, solapamiento) in [("Breve", BREVE), ("Amplio", AMPLIO)]:
    divisor = RecursiveCharacterTextSplitter(
        chunk_size=tamano,
        chunk_overlap=solapamiento,
        add_start_index=True,
    )
    fragmentos = divisor.split_documents(documentos)
    print(f"{etiqueta}: {len(fragmentos)} fragmentos (tamaño {tamano}, solapamiento {solapamiento})")

    nombre_coleccion = f"comparacion_{etiqueta.lower()}"
    indice = Chroma(collection_name=nombre_coleccion, embedding_function=embeddings)
    indice.delete_collection()  # evita duplicados si se repite la celda
    indice = Chroma(collection_name=nombre_coleccion, embedding_function=embeddings)
    indice.add_documents(documents=fragmentos)
    indices_temporales[etiqueta] = indice

for etiqueta, indice in indices_temporales.items():
    print(f"\n=== {etiqueta} ===")
    hallazgos = indice.similarity_search_with_score(query=PREGUNTA_COMPARACION, k=app.K)

    for numero, (documento, distancia) in enumerate(hallazgos, start=1):
        archivo = Path(documento.metadata["source"]).name
        pagina = documento.metadata["page"] + 1
        inicio = documento.metadata["start_index"]
        print(f"[{numero}] {archivo}, página {pagina}, desde el carácter {inicio}, distancia {distancia:.3f}")
        print(documento.page_content)
        print()

Breve: 560 fragmentos (tamaño 500, solapamiento 100)
Amplio: 280 fragmentos (tamaño 1000, solapamiento 200)

=== Breve ===
[1] ley_24449_transito.pdf, página 29, desde el carácter 809, distancia 0.649
Ley N° 25.965
 (
/normativa/nacional/norma-102148
)
 B.O.
21/12/2004).
ARTICULO  41.
 —  PRIORIDADES.  Todo  conductor  debe  ceder  siempre  el  paso  en  las
encrucijadas al que cruza desde su derecha. Esta prioridad del que viene por la derecha es
absoluta, y sólo se pierde ante:
a) La señalización específica en contrario;
b) Los vehículos ferroviarios;
c) Los vehículos del servicio público de urgencia, en cumplimiento de su misión;

[2] ley_24449_transito.pdf, página 31, desde el carácter 1222, distancia 0.789
a) Los vehículos deben:
1. Con luz verde a su frente, avanzar;
2. Con luz roja, detenerse antes de la línea marcada a tal efecto o de la senda peatonal,
evitando luego cualquier movimiento;
3. Con luz amarilla, detenerse si se estima que no se alcanzará a transponer la encrucija

### Actividad 1 — Elegir un corte

Del análisis empírico realizado sobre las distintas estrategias de particionamiento del corpus, se evaluó el comportamiento de cortes de menor extensión (500 caracteres con solapamiento de 100) frente a los de mayor extensión (1000 caracteres con solapamiento de 200). 

* **Selección del parámetro óptimo:** Se determina que el tamaño de 1000 caracteres con solapamiento de 200 resulta más adecuado para este dominio normativo analizado. 
* **Justificación técnica:** Este tamaño mayor permite conservar la integridad estructural de los artículos de la Ley de Tránsito y sus incisos reglamentarios dentro de un mismo bloque de recuperación, intentando evitar cortes abruptos que fragmenten la justificación.
* **Riesgo observado:** El fragmento amplio incluye algo de texto vecino no referido a la consulta, y el modelo puede citar fragmentos que no neesariamente respaldan la respuesta.

## 3. Probar el sistema completo

Escribir los casos antes de ejecutarlos: la pregunta y la fuente esperada (archivo y página). Mínimo exigido: una pregunta respondible por documento y dos cuyo dato no figure en el corpus. 

In [13]:

CASOS = [
    {"tipo": "Respondible 1", "pregunta": "Quién tiene prioridad en las rotondas?", "esperada": "Ley 24.449, Artículo 43 (inciso e), página 31. Establece que tiene prioridad de paso el que circula por la rotonda sobre el que intenta ingresar, y debe ceder el paso el que egresa."},
    {"tipo": "Respondible 2", "pregunta": "Cuándo entró en vigencia la ley?", "esperada": "decreto_779_1995_reglamentario.pdf, página 4"},
    {"tipo": "Respondible 3", "pregunta": "Sobre quién se aplica el descuento de puntos del scoring en una infracción?", "esperada": "decreto_242_2022_scoring.pdf, art. 2° (página 4): la deducción de puntos se aplica cuando el conductor profesional es también titular registral del dominio"},
    {"tipo": "Respondible 4", "pregunta": "Cual es la cantidad de UF a descontar por cruzar cuando el semaforo está en rojo?", "esperada": "anexo2_decreto_779_1995.pdf, art. 57 (página 8): 5 puntos por cruzar con el semáforo en rojo"},
    {"tipo": "Sin respuesta 1", "pregunta": "¿Cuánto cuesta renovar la licencia de conducir?", "esperada": "No figura en el corpus"},
    {"tipo": "Sin respuesta 2", "pregunta": "¿Qué porcentaje de descuento se aplica por pago voluntario de infracciones en la provincia de Córdoba?", "esperada": "No figura en el corpus"},
]

for caso in CASOS:
    assert "COMPLETAR" not in caso["pregunta"], f"Falta la pregunta de {caso['tipo']}."
    assert "COMPLETAR" not in caso["esperada"], f"Falta la fuente esperada de {caso['tipo']}."

# Abre el índice de la aplicación (lo crea la primera vez, puede tardar)
app.INDICE = app.preparar_indice()
print("Fragmentos en el índice de la aplicación:", len(app.INDICE.get()["ids"]))


Fragmentos en el índice de la aplicación: 280


In [12]:
def limpiar(texto):
    # Una fila de tabla Markdown no admite saltos de línea ni barras verticales
    return " ".join(str(texto).replace("|", "/").split())


filas = []

for caso in CASOS:
    hallazgos = app.INDICE.similarity_search_with_score(query=caso["pregunta"], k=app.K)
    recuperados = []

    for documento, distancia in hallazgos:
        archivo = Path(documento.metadata["source"]).name
        recuperados.append(f"{archivo} p. {documento.metadata['page'] + 1}")

    respuesta, _ = app.consultar(caso["pregunta"])

    print(f"\n{caso['tipo']}: {caso['pregunta']}")
    print("  Fuente esperada:", caso["esperada"])
    print("  Recuperado:", "; ".join(recuperados))
    print("  Respuesta:", respuesta)

    filas.append([
        caso["tipo"],
        limpiar(caso["pregunta"]),
        limpiar(caso["esperada"]),
        limpiar("; ".join(recuperados)),
        limpiar(respuesta),
        "(completar)",
    ])

print("\n--- Tabla para pegar en el README (completá la última columna) ---\n")
print("| Caso | Pregunta | Fuente esperada | Fragmento recuperado | Respuesta del modelo | Qué observé |")
print("| --- | --- | --- | --- | --- | --- |")

for fila in filas:
    print("| " + " | ".join(fila) + " |")


Respondible 1: Quién tiene prioridad en las rotondas?
  Fuente esperada: Ley 24.449, Artículo 43 (inciso e), página 31. Establece que tiene prioridad de paso el que circula por la rotonda sobre el que intenta ingresar, y debe ceder el paso el que egresa.
  Recuperado: ley_24449_transito.pdf p. 29; ley_24449_transito.pdf p. 31; ley_24449_transito.pdf p. 30
  Respuesta: En las rotondas, la prioridad de paso es para el que circula por ella sobre el que intenta ingresar, debiendo ceder la zona central al que egresa, salvo señalización en contrario. ([2])

Respondible 2: Cuándo entró en vigencia la ley?
  Fuente esperada: decreto_779_1995_reglamentario.pdf, página 4
  Recuperado: decreto_779_1995_reglamentario.pdf p. 4; ley_24449_transito.pdf p. 68; ley_24449_transito.pdf p. 69
  Respuesta: La Ley N° 24.449 y su reglamentación entraron en vigencia el 1° de Diciembre de 1995 ([1], [2]).

Respondible 3: Sobre quién se aplica el descuento de puntos del scoring en una infracción?
  Fuente espe

### Metodología de validación cruzada y evaluación de recuperación

Esta etapa comprende la auditoría empírica de los resultados obtenidos frente a las fuentes documentales:
* **Verificación de recuperación:** Contrastación de los fragmentos recuperados por el sistema con las páginas originales de los archivos PDF del corpus.
* **Evaluación de generación y fidelidad:** Control de correspondencia semántica entre la respuesta generada por el modelo y el contenido estricto de la fuente.
* **Análisis de abstención:** Comprobación del comportamiento del sistema ante consultas no respondibles por el corpus, considerando que el índice vectorial basado en distancia (Chroma) retorna vecinos por proximidad geométrica independientemente de su relevancia semántica real.
* **Registro documental:** Consolidación de las observaciones cualitativas y de trazabilidad en la Sección 6 de la documentación técnica (`README.md`).

## 4. Diagnóstico: de dónde viene una falla

Se evaluó el comportamiento operativo del sistema frente a un conjunto de consultas representativas del dominio normativo, auditando la precisión en la recuperación de artículos específicos, la fidelidad de la respuesta generada y el manejo de restricciones ante vacíos normativos. El detalle pormenorizado de los casos analizados, los diagnósticos de recuperación y las pruebas de sensibilidad se encuentran consolidados en la **Sección 6 del `README.md`**.


In [ ]:
PREGUNTA_K = CASOS[2]["pregunta"]  # cambiar por el caso que se quiera diagnosticar

for k in (1, 3, 5):
    print(f"\nk = {k}")
    hallazgos = app.INDICE.similarity_search_with_score(query=PREGUNTA_K, k=k)

    for numero, (documento, distancia) in enumerate(hallazgos, start=1):
        archivo = Path(documento.metadata["source"]).name
        pagina = documento.metadata["page"] + 1
        print(f"  [{numero}] {archivo}, página {pagina}, distancia {distancia:.3f}")


k = 1
  [1] decreto_242_2022_scoring.pdf, página 3, distancia 0.812

k = 3
  [1] decreto_242_2022_scoring.pdf, página 3, distancia 0.812
  [2] ley_24449_transito.pdf, página 61, distancia 0.896
  [3] anexo2_decreto_779_1995.pdf, página 10, distancia 0.903

k = 5
  [1] decreto_242_2022_scoring.pdf, página 3, distancia 0.812
  [2] ley_24449_transito.pdf, página 61, distancia 0.896
  [3] anexo2_decreto_779_1995.pdf, página 10, distancia 0.903
  [4] anexo2_decreto_779_1995.pdf, página 10, distancia 0.907
  [5] anexo2_decreto_779_1995.pdf, página 9, distancia 0.911


**Si se quiere ver cómo cambia la respuesta generada con otro `k`**, no alcanza con cambiarlo desde acá: hay que editar `K` en `app.py` y reiniciar el kernel. El prompt arma sus etiquetas `[1]`, `[2]`... al importar el módulo; si cambia `K` después, el prompt y el contexto quedan desalineados.

Si cambia `TAMANO_FRAGMENTO` o `SOLAPAMIENTO` en `app.py`, se crea una colección nueva y el primer arranque vuelve a indexar (con `python app.py --reindexar` se reconstruye la colección actual).